# 07 — Tiered Memory Architecture (Issue 9)

Demonstrates the tiered memory system that manages LLM context across simulation days:
1. Load agents with accumulated reflections (from prior phases)
2. Compress daily memories → daily summaries (2–3 sentences)
3. Compress weekly memories → weekly summaries (2–3 sentences)
4. Assemble full context and inspect structure at different day offsets
5. Verify persona drift mitigation (reinforcement every N days)

**Covers:** Issue 9 (Tiered Memory Architecture)  
**Depends on:** Issue 6

In [1]:
import os, sys, random
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import ClimatePolicyID, SURVEY_QUESTIONS
from cag.io.llm import load_api_key

# Attribute maps and IDs
from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

print("Imports OK")

Imports OK


## 1. Load Data & Build Environment

In [2]:
random.seed(42)
year = 2026

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load("../data/yougov_survey_data/YouGovProcessedData_train.csv")

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")
sn.assign_political_exposure()
sn.create_network(seed=42)
sn.assign_network_blocks()

target_policy = ClimatePolicyID.CARBON_TAX

print(f"Total citizens: {len(sn.agents_active)}")

Total citizens: 651


## 2. Inject Synthetic Reflections (Multi-Day)

To test memory compression without running expensive LLM phases,
inject mock reflections spanning several days for a single demo citizen.

In [ ]:
citizen = list(sn.agents_active.values())[0]

citizen.reflections = [
    # Day 1
    {"day": 1, "phase": "P-A", "policy_id": target_policy, "text": "The message about insulating homes to cut bills really resonated with me. I see families at school struggling with heating costs every winter and this feels like something that could genuinely help them while also being good for the planet.", "messages_received": ["message_1"]},
    {"day": 1, "phase": "P-B", "policy_id": target_policy, "text": "I hadn't really considered how much the green transition costs ordinary working people. The point about green levies pushing up energy bills gave me real pause — some of my pupils' parents are already choosing between heating and eating.", "messages_received": ["message_2"]},
    {"day": 1, "phase": "C", "policy_id": target_policy, "text": "Most people I spoke with today seemed broadly supportive of climate action but genuinely worried about who ends up paying for it. One person made a fair point that a flat carbon tax hits everyone equally regardless of their income.", "messages_received": ["message_3", "message_4"]},

    # Day 2
    {"day": 2, "phase": "P-A", "policy_id": target_policy, "text": "The wealth tax argument is genuinely compelling to me. If the wealthiest people and corporations are causing the most pollution then it makes complete sense that they should be the ones funding the solution rather than ordinary taxpayers.", "messages_received": ["message_5"]},
    {"day": 2, "phase": "P-B", "policy_id": target_policy, "text": "The fracking argument does not convince me at all but I think the broader point about energy sovereignty is actually worth taking seriously. We cannot be completely dependent on imports from unstable regions for something so essential.", "messages_received": ["message_6"]},
    {"day": 2, "phase": "C", "policy_id": target_policy, "text": "Someone made an excellent point today that even if we support the carbon tax in principle the implementation details matter enormously. A badly designed tax with no exemptions could genuinely hurt the very people it is supposed to protect.", "messages_received": ["message_7"]},

    # Day 3
    {"day": 3, "phase": "P-A", "policy_id": target_policy, "text": "I am increasingly persuaded that renewables are now genuinely cheaper than fossil fuels. The argument that it is actually our dependence on volatile gas and oil markets making people poor rather than green policies is a powerful reframing.", "messages_received": ["message_8"]},
    {"day": 3, "phase": "P-B", "policy_id": target_policy, "text": "The war on drivers rhetoric feels deliberately manipulative to me. But I noticed it was emotionally effective — I caught myself feeling irritated about speed limits near my school before remembering they exist for child safety.", "messages_received": ["message_9"]},
    {"day": 3, "phase": "C", "policy_id": target_policy, "text": "A couple of people were noticeably more skeptical today than before. One said they had turned against the carbon tax entirely because they simply do not trust the government to spend the revenue properly. That concern feels legitimate to me.", "messages_received": ["message_10", "message_11"]},

    # Day 4
    {"day": 4, "phase": "P-A", "policy_id": target_policy, "text": "The redistribution framing keeps coming back to me and I find it increasingly persuasive. Tax the billionaires, insulate homes, lower bills for working families. It is simple and coherent and I think I support it more strongly now than before.", "messages_received": ["message_12"]},
    {"day": 4, "phase": "P-B", "policy_id": target_policy, "text": "Today's message focused on solar farms ruining the British countryside. I live in a city so this does not affect me directly but I can understand why rural communities might feel strongly about it. Still, unchecked climate change will do far worse.", "messages_received": ["message_13"]},
    {"day": 4, "phase": "C", "policy_id": target_policy, "text": "The group felt noticeably more polarised today than earlier in the week. The skeptics have dug in and the supporters are more vocal. I find myself firmly in the supportive camp but I am trying hard to take the cost concerns seriously.", "messages_received": ["message_14", "message_15"]},

    # Day 5
    {"day": 5, "phase": "P-A", "policy_id": target_policy, "text": "I think I have now heard enough arguments to feel genuinely confident in my position. The transition will be difficult and disruptive but doing nothing is clearly worse. The carbon tax is imperfect but it is a reasonable starting point for action.", "messages_received": ["message_16"]},
    {"day": 5, "phase": "P-B", "policy_id": target_policy, "text": "The Net Zero is Net Poverty slogan is catchy but I think it is intellectually dishonest. Climate inaction is the real route to widespread poverty and suffering. I am finding it increasingly difficult to take these anti-climate messages seriously.", "messages_received": ["message_17"]},
    {"day": 5, "phase": "C", "policy_id": target_policy, "text": "Today's discussion felt like something of a turning point. Even the most skeptical people in the group acknowledged that some form of action is needed — they just disagree strongly about the method. There is more common ground than the rhetoric suggests.", "messages_received": ["message_18", "message_19"]},
]

# Seed opinion history for a few policies
citizen.opinion_history = {
    ClimatePolicyID.CARBON_TAX: [(0, 1), (1, 1), (2, 2), (3, 2), (4, 2), (5, 3)],
    ClimatePolicyID.RENEWABLE_ENERGY: [(0, 0), (1, 1), (2, 1), (3, 1), (4, 2), (5, 2)],
}

## 3. Daily Memory Compression

Compress reflections from day d-2 into a 2–3 sentence daily summary.

In [4]:
day = 3
memory = citizen.compress_daily_memory(day=day, policy_id=target_policy, api_key=load_api_key("openai"), model="gpt-4o-mini", provider="openai")
print(memory)

I'm becoming convinced that renewables are genuinely cheaper than fossil fuels, and it's compelling to consider our reliance on volatile gas and oil markets as the real source of economic hardship, rather than green policies. While I find the rhetoric against drivers to be manipulative, I felt its emotional impact, and I also noticed some skepticism about the carbon tax, which resonates with me due to valid concerns about government accountability.


In [5]:
# Run memory management
for d in range(1, 6):
    memory = citizen.manage_memory(day=d, policy_id=target_policy, api_key=load_api_key("openai"), model="gpt-4o-mini", provider="openai")

## 4. Weekly Memory Compression

Compress daily summaries from week 1 (days 1–7) into a weekly summary.

In [12]:
# TODO: Call compress_weekly_memory() for week 1
# Show the weekly summary text

## 5. Context Assembly at Different Days

Inspect what `assemble_context(day)` produces at various simulation days:
- Day 1: persona + current reflections only
- Day 3: persona + full reflections (days 2–3)
- Day 5: persona + daily summary (day 3) + full reflections (days 4–5)
- Day 10: persona + weekly summary (week 1) + daily summaries + recent reflections

In [6]:
# TODO: Print assemble_context(day) for days 1, 3, 5, 10
# Show the full context string with section labels
for d in range(1,6):
    context = citizen.assemble_context(day=d, policy_id=target_policy)
    print(f"Context for day {d}:\n{context}\n{'-'*50}\n")

Context for day 1:
I am a 41 year old female living in the North West. My ethnicity is white. I have a University or CNAA first degree (e.g. BA, B.Sc, B.Ed). My gross household income is £70,000 - £99,999 per year. I am a parent. I position myself slightly right-of-centre of the political spectrum. I voted for the Liberal Democrat party candidate in the 2019 General Election. I voted to remain in the 2016 EU Referendum.
When it comes to my core values and worldview: I care about the people close to me and have a basic respect for nature, but I do not actively champion global equality or make environmental protection a primary, driving life focus. I am not strongly driven by the need to get ahead of others, impress people, or hold leadership positions where I tell others what to do. I prefer routine and the familiar, showing little interest in taking risks, seeking out new adventures, or coming up with highly original ideas. I maintain a general respect for elders and standard societal 

## 6. Opinion Trajectory in Context

In [ ]:


# Print the trajectory string

# Verify it appears in context
ctx = citizen.assemble_context(day=5, policy_id=target_policy)

print(ctx)

Opinion trajectory:

Please say how much you support or oppose government policie: Day 0: E, Day 1: E, Day 2: F, Day 3: F, Day 4: F, Day 5: G
Please say how much you support or oppose government policie: Day 0: D, Day 1: E, Day 2: E, Day 3: E, Day 4: F, Day 5: F
I am a 41 year old female living in the North West. My ethnicity is white. I have a University or CNAA first degree (e.g. BA, B.Sc, B.Ed). My gross household income is £70,000 - £99,999 per year. I am a parent. I position myself slightly right-of-centre of the political spectrum. I voted for the Liberal Democrat party candidate in the 2019 General Election. I voted to remain in the 2016 EU Referendum.
When it comes to my core values and worldview: I care about the people close to me and have a basic respect for nature, but I do not actively champion global equality or make environmental protection a primary, driving life focus. I am not strongly driven by the need to get ahead of others, impress people, or hold leadership posit

## 7. Persona Drift Mitigation

Verify persona reinforcement appears at end of context every N days.

In [ ]:
# TODO: Check persona reinforcement at day 5 and day 10
# Verify key persona attributes are repeated at end of context

## 8. Context Token Length Analysis

Track how context length grows with and without compression.

In [ ]:
# TODO: Compare context word/token counts across days
# - Without compression: raw reflections accumulate
# - With compression: bounded by tiered summaries

## 9. Sanity Checks

In [ ]:
# TODO: Sanity checks
# - Daily summaries exist for compressed days
# - Weekly summaries exist for completed weeks
# - Context at day 1 contains no summaries, just persona + reflections
# - Context grows sub-linearly with number of days
# - Persona reinforcement interval is respected